<a href="https://colab.research.google.com/github/dakshatakamde46-creator/Dynamic-Chatbot/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q langchain langchain-community langchain-openai chromadb schedule pydantic


In [ ]:
pip install -q sentence-transformers


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


DB_DIR = "./chroma_db"


/tmp/ipykernel_592/4173068831.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
FLIGHT_PROJECT_LOGS = [
    {"id": "log_881", "text": "AeroEngine Update: Next-gen turbine cooling trials are rescheduled for mid-August 2026.", "stream": "Propulsion_Lab"},
    {"id": "log_882", "text": "Telemetry systems must now route primary sensor telemetry packets through port 8443.", "stream": "Avionics_Division"},
]

def capture_stream_updates():
    """Extracts raw telemetry logs and formats them into structured LangChain components."""
    print(f"[{datetime.now()}] Scanning external aerospace streams for updates...")
    processed_records = []

    for entry in FLIGHT_PROJECT_LOGS:
        record = Document(
            page_content=entry["text"],
            metadata={"origin": entry["stream"], "record_id": entry["id"]}
        )
        processed_records.append(record)
    return processed_records

def sync_knowledge_pipeline():
    """Processes incoming streams, chunks text parameters, and synchronizes the Chroma matrix."""
    incoming_data = capture_stream_updates()
    if not incoming_data:
        print("No fresh log data detected.")
        return
    segmenter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=40)
    tokenized_chunks = segmenter.split_documents(incoming_data)


    knowledge_matrix = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    knowledge_matrix.add_documents(tokenized_chunks)
    print(f"Success: Indexed {len(tokenized_chunks)} structural chunks into the vector store.")

In [ ]:
schedule.clear()
schedule.every(1).minutes.do(sync_knowledge_pipeline)

print("Automated Sync Matrix Initated. Monitoring stream nodes... (Press STOP to terminate)")
try:
    sync_knowledge_pipeline() # Execute baseline injection
    for _ in range(3):
        schedule.run_pending()
        time.sleep(1)
except KeyboardInterrupt:
    print("\nBackground sync pipeline suspended by operator.")

Automated Sync Matrix Initated. Monitoring stream nodes... (Press STOP to terminate)
[2026-07-13 12:56:02.801166] Scanning external aerospace streams for updates...
Success: Indexed 2 structural chunks into the vector store.


In [ ]:
knowledge_matrix = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
neural_retriever = knowledge_matrix.as_retriever(search_kwargs={"k": 1})
target_inquiry = "When are the next-gen turbine cooling trials scheduled?"
matched_nodes = neural_retriever.invoke(target_inquiry)
print(f"User Query: {target_inquiry}\n" + "="*50)
if matched_nodes:
    print(f"Extracted Knowledge: {matched_nodes[0].page_content}")
    print(f"Metadata Footprint: {matched_nodes[0].metadata}")
else:
    print("Target context vector not found.")

User Query: When are the next-gen turbine cooling trials scheduled?
Extracted Knowledge: AeroEngine Update: Next-gen turbine cooling trials are rescheduled for mid-August 2026.
Metadata Footprint: {'origin': 'Propulsion_Lab', 'record_id': 'log_881'}
